The **Pandas API on Spark** (`pyspark.pandas`) lets you write familiar pandas code that runs on a distributed Spark cluster. Instead of learning an entirely new API, you import `pyspark.pandas as ps` and use the same DataFrame and Series interfaces you already know — but backed by Spark's execution engine.

In a banking context this matters when a pandas workflow that worked on a sample of 100,000 transactions breaks down on 500 million rows. The Pandas API on Spark scales the same code without a full rewrite.

In this notebook you will learn:
- How the Pandas API on Spark relates to native Spark DataFrames and plain pandas
- Creating and working with `ps.DataFrame` and `ps.Series`
- Converting freely between pandas, Spark, and pandas-on-Spark
- Index behaviour and the `default_index_type` setting
- Limitations to know before the exam

## The Three DataFrame APIs

| | `pandas` | `pyspark.sql` (Spark) | `pyspark.pandas` |
|---|---|---|---|
| Scale | Single machine | Distributed cluster | Distributed cluster |
| API style | pandas | Spark SQL / DSL | pandas |
| Lazy evaluation | No | Yes | Yes (translated to Spark) |
| Row ordering guaranteed | Yes | No | No |
| Import | `import pandas as pd` | `from pyspark.sql import ...` | `import pyspark.pandas as ps` |

`pyspark.pandas` translates your pandas-style calls into Spark logical plans and executes them across the cluster. The surface area is familiar; the engine is Spark.

## Setup

In [ ]:
import os
import pandas as pd
import pyspark.pandas as ps
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, BooleanType,
    DecimalType, DoubleType, IntegerType, DateType
)

spark = (
    SparkSession.builder
    .appName("PandasAPIOnSpark")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

# pyspark.pandas needs access to the active SparkSession — already set above
# Suppress the default_index_type warning for the examples below
ps.set_option("compute.default_index_type", "distributed")

DATA = os.path.join(os.path.dirname(os.path.abspath("22-pandas-api-on-spark.ipynb")), "data")

# Load raw data as plain Spark DataFrames for setup
customers_sdf = spark.read.schema(
    StructType([
        StructField("customer_id",  StringType(),  False),
        StructField("full_name",    StringType(),  True),
        StructField("city",         StringType(),  True),
        StructField("credit_score", IntegerType(), True),
        StructField("kyc_verified", BooleanType(), True),
    ])
).csv(f"{DATA}/customers.csv", header=True)

card_txns_sdf = spark.read.schema(
    StructType([
        StructField("txn_id",      StringType(),  False),
        StructField("customer_id", StringType(),  True),
        StructField("amount",      DoubleType(),  True),
        StructField("merchant",    StringType(),  True),
        StructField("status",      StringType(),  True),
        StructField("is_fraud",    BooleanType(), True),
    ])
).csv(f"{DATA}/card_transactions.csv", header=True)

print("Spark DataFrames loaded.")

## Creating Pandas-on-Spark DataFrames

Three patterns — from a plain pandas DataFrame, from a Spark DataFrame, or directly from data.

In [ ]:
# 1. From a plain pandas DataFrame
pdf = pd.DataFrame({
    "customer_id":  ["CUST0001", "CUST0002", "CUST0003"],
    "city":         ["Mumbai",   "Delhi",    "Bangalore"],
    "credit_score": [720,        680,        750],
})
psdf_from_pandas = ps.from_pandas(pdf)
print(type(psdf_from_pandas))   # pyspark.pandas.frame.DataFrame
psdf_from_pandas

In [ ]:
# 2. From a Spark DataFrame — to_pandas_on_spark() is the idiomatic bridge
customers_psdf = customers_sdf.to_pandas_on_spark()
print(type(customers_psdf))     # pyspark.pandas.frame.DataFrame
customers_psdf.head(3)

In [ ]:
# 3. Directly from Python data — identical to pd.DataFrame() syntax
ps.DataFrame({
    "product": ["savings", "current", "loan"],
    "count":   [3200, 1800, 950],
})

## Common Operations — Familiar Pandas Syntax

Column selection, filtering, and descriptive statistics work identically to pandas.

In [ ]:
card_psdf = card_txns_sdf.to_pandas_on_spark()

# Column selection
card_psdf[["txn_id", "amount", "status"]].head(5)

In [ ]:
# Boolean filtering — same syntax as pandas
fraud = card_psdf[card_psdf["is_fraud"] == True]
print("Fraud transactions:", len(fraud))
fraud[["txn_id", "amount", "merchant"]].head(5)

In [ ]:
# Descriptive stats — runs as a distributed Spark job
card_psdf[["amount"]].describe()

In [ ]:
# value_counts — equivalent to groupBy().count() in Spark
card_psdf["status"].value_counts()

## GroupBy and Aggregation

In [ ]:
# groupby — same syntax as pandas groupby
summary = (
    card_psdf
    .groupby("status")["amount"]
    .agg(["count", "sum", "mean"])
    .rename(columns={"count": "txn_count", "sum": "total_amount", "mean": "avg_amount"})
    .reset_index()
)
summary

In [ ]:
# groupby with transform — assign group-level stat back to each row
card_psdf["pct_of_group_total"] = (
    card_psdf["amount"] / card_psdf.groupby("status")["amount"].transform("sum") * 100
).round(2)

card_psdf[["txn_id", "status", "amount", "pct_of_group_total"]].head(8)

## Merge (Join)

`ps.merge()` supports the same join types as `pd.merge()` — inner, left, right, outer.

In [ ]:
# Enrich transactions with customer city
enriched = ps.merge(
    card_psdf[["txn_id", "customer_id", "amount", "is_fraud"]],
    customers_psdf[["customer_id", "city", "credit_score"]],
    on="customer_id",
    how="left",
)
enriched.head(6)

## Converting Between APIs

Moving data between pandas, pandas-on-Spark, and native Spark DataFrames is explicit and low-friction.

```
pandas DataFrame  ──────────────────────────────────────────────────  Spark DataFrame
       │  ps.from_pandas(pdf)              spark.createDataFrame(pdf)  │
       │  ────────────────────►  ps.DataFrame  ◄───────────────────── │
       │                           │       │         .to_spark()       │
       │  .to_pandas()             │       └──────────────────────────►│
       │  ◄────────────────────────┘
```

In [ ]:
# pandas → Spark DataFrame (two ways)
pdf_sample = pdf  # plain pandas DataFrame from earlier

# Option A — via spark.createDataFrame (explicit, standard)
sdf_a = spark.createDataFrame(pdf_sample)
sdf_a.printSchema()

# Option B — via ps.from_pandas then .to_spark()
sdf_b = ps.from_pandas(pdf_sample).to_spark()
sdf_b.show()

print(type(sdf_a))  # pyspark.sql.dataframe.DataFrame
print(type(sdf_b))  # pyspark.sql.dataframe.DataFrame

In [ ]:
# Spark DataFrame → pandas (collects to driver — only for small results)
pdf_result = customers_sdf.limit(5).toPandas()
print(type(pdf_result))  # pandas.core.frame.DataFrame
pdf_result

In [ ]:
# pandas-on-Spark → native Spark DataFrame
sdf_from_psdf = customers_psdf.to_spark()
print(type(sdf_from_psdf))  # pyspark.sql.dataframe.DataFrame
sdf_from_psdf.show(3)

# pandas-on-Spark → plain pandas (collects to driver)
pdf_from_psdf = customers_psdf.head(3).to_pandas()
print(type(pdf_from_psdf))  # pandas.core.frame.DataFrame

## Index Behaviour and `default_index_type`

pandas guarantees a meaningful integer index. Distributed DataFrames do not — rows live across partitions with no natural global order. `pyspark.pandas` offers three index strategies controlled by `ps.set_option`:

| `default_index_type` | Behaviour | Cost |
|---|---|---|
| `"sequence"` | Assigns 0, 1, 2, … in order (default in early versions) | Requires a full sort — expensive |
| `"distributed-sequence"` | Consecutive integers but computed per-partition in parallel | Cheaper than sequence, still one pass |
| `"distributed"` | Non-consecutive partition-local IDs | Zero extra cost — recommended |

Set `"distributed"` unless you need positional integer indexing.

In [ ]:
# Change index type globally
ps.set_option("compute.default_index_type", "distributed")

psdf_dist = card_txns_sdf.to_pandas_on_spark()
print(psdf_dist.index[:5].to_list())   # non-consecutive integers — zero cost

# Or set per-DataFrame via index_col when converting from Spark
psdf_indexed = card_txns_sdf.to_pandas_on_spark(index_col="txn_id")
psdf_indexed.head(3)   # txn_id is now the index

## Execution Model

Every `pyspark.pandas` operation is translated into a Spark logical plan and executed lazily. A result is materialised (a Spark job fires) only when you call an action:

- `.show()` / `.head()` / `.display()`
- `.to_pandas()` / `.toPandas()`
- `.count()` / `.len()`
- Any aggregation that returns a scalar

This means chaining transformations is free — only the final action triggers computation.

## When to Use Each API

| Scenario | Recommended API |
|---|---|
| Data fits in memory on one machine | `pandas` — simplest, full pandas feature set |
| New Spark code, performance matters | `pyspark.sql` — full control, best performance |
| Migrating existing pandas pipeline to Spark scale | `pyspark.pandas` — minimal code change |
| Exam question about pandas syntax on Spark | `pyspark.pandas` — `import pyspark.pandas as ps` |
| Vectorized row-wise transformation in Spark | `@pandas_udf` (see notebook 11) |

On Databricks, `pyspark.pandas` is the recommended way to scale pandas workloads — it replaces the legacy `koalas` package.

## Limitations

Not all pandas operations are supported or behave identically:

- **No guaranteed row order** — `sort_values` produces a correct sort but the index is not stable across runs without `index_col`.
- **`apply` with `axis=1` (row-wise)** is supported but slow — internally serialises each row to Python. Prefer column-wise `apply` or use `@pandas_udf`.
- **`inplace=True`** is not supported — always assign the result to a new variable.
- **`iterrows` / `itertuples`** — not supported. These collect to driver and defeat the purpose.
- **Multi-index** — limited support. Prefer flat indexes.
- **`iloc` with arbitrary slicing** — restricted because rows have no total order without a sequence index.

When you hit an unsupported operation, convert to a native Spark DataFrame, perform the operation with Spark SQL functions, then convert back.

## Summary

- `import pyspark.pandas as ps` gives you a pandas-compatible API backed by Spark — scale without a full rewrite.
- Create `ps.DataFrame` from plain pandas (`ps.from_pandas(pdf)`), from a Spark DataFrame (`.to_pandas_on_spark()`), or directly from Python data.
- Convert back: `.to_spark()` → Spark DataFrame; `.to_pandas()` → plain pandas (collects to driver).
- `spark.createDataFrame(pdf)` converts a plain pandas DataFrame to a native Spark DataFrame directly.
- Set `ps.set_option("compute.default_index_type", "distributed")` to avoid expensive global sorting for the index.
- Operations are lazy — only actions trigger a Spark job.
- Avoid `inplace=True`, `iterrows`, and row-wise `apply` — use native Spark functions or `@pandas_udf` instead.
- On Databricks, `pyspark.pandas` replaces the legacy `koalas` library.